In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

# Silver Script
---

In [0]:
# Access ADLS Gen 2 (Storage Account) from Databricks
# refer documentation: https://docs.azure.cn/en-us/databricks/connect/storage/azure-storage
spark.conf.set("fs.azure.account.auth.type.adwhmk31.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.adwhmk31.dfs.core.windows.net", 
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.adwhmk31.dfs.core.windows.net", "<application_id>")
spark.conf.set("fs.azure.account.oauth2.client.secret.adwhmk31.dfs.core.windows.net", "<secret_scrednetial>")
spark.conf.set("fs.azure.account.oauth2.client.endpoint.adwhmk31.dfs.core.windows.net", 
               "https://login.microsoftonline.com/<directory_id>/oauth2/token")

### 1. Data Loading

In [0]:
df_cal=spark.read.format('csv').option("header",True).option("inferSchema",True).load('abfss://bronze@adwhmk31.dfs.core.windows.net/calendar')

In [0]:
df_cus=spark.read.format('csv').option("header",True).option("inferSchema",True).load('abfss://bronze@adwhmk31.dfs.core.windows.net/customers')

In [0]:
df_procat=spark.read.format('csv').option("header",True).option("inferSchema",True).load('abfss://bronze@adwhmk31.dfs.core.windows.net/product_categories')

In [0]:
df_pro =spark.read.format('csv').option("header",True).option("inferSchema",True).load('abfss://bronze@adwhmk31.dfs.core.windows.net/products')

In [0]:
df_ret=spark.read.format('csv').option("header",True).option("inferSchema",True).load('abfss://bronze@adwhmk31.dfs.core.windows.net/returns')

In [0]:
df_sales_2015 = spark.read.format("csv").option("header", True).load("abfss://bronze@adwhmk31.dfs.core.windows.net/sales_2015")


In [0]:
df_sales_2016= spark.read.format("csv").option("header", True).load("abfss://bronze@adwhmk31.dfs.core.windows.net/sales_2016")


In [0]:

df_sales_2017 =spark.read.format("csv").option("header", True).load("abfss://bronze@adwhmk31.dfs.core.windows.net/sales_2017")


In [0]:
df_ter=spark.read.format('csv').option("header",True).option("inferSchema",True).load('abfss://bronze@adwhmk31.dfs.core.windows.net/territories')

In [0]:
df_subcat=spark.read.format('csv').option("header",True).option("inferSchema",True).load('abfss://bronze@adwhmk31.dfs.core.windows.net/product_subcategories')

In [0]:
df_sales = df_sales_2015.unionByName(df_sales_2016).unionByName(df_sales_2017)


### 2. Transformations

##### calender

In [0]:
df_cal = (
    df_cal
    .withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
    .withColumn("Year", year(col("Date")))
    .withColumn("Month", month(col("Date")))
    .withColumn("MonthName", date_format(col("Date"), "MMMM"))
    .withColumn("Quarter", quarter(col("Date")))
    .withColumn("DayOfWeek", dayofweek(col("Date")))
    .dropDuplicates()
)

In [0]:
df_cal.write.format('parquet').mode('overwrite').save("abfss://silver@adwhmk31.dfs.core.windows.net/calendar")

##### customers

In [0]:
df_cus=(df_cus
    .withColumn('BirthDate', to_date(col('BirthDate'), 'yyyy-MM-dd'))
    .withColumn('AnnualIncome', regexp_replace(col('AnnualIncome'), '[$,]', '').cast('decimal(12,2)'))
    .withColumn('HomeOwner', when(col('HomeOwner') == 'Y', True).otherwise(False))
    .withColumn('Gender', when(col('Gender') == 'NA', None).otherwise(col('Gender')))
    .withColumn('Prefix', when(col('Prefix') == 'NA', None).otherwise(col('Prefix')))
    .withColumn('FirstName', initcap(col('FirstName')))
    .withColumn('LastName', initcap(col('LastName')))
    .withColumn('FullName', concat_ws(" ", col('Prefix'), col('FirstName'), col('LastName')))
    .dropDuplicates()
)

In [0]:
df_cus.write.format('parquet').mode('overwrite').save("abfss://silver@adwhmk31.dfs.core.windows.net/customers")

##### product subcategories

In [0]:
df_subcat.write.format('parquet').mode('overwrite').save("abfss://silver@adwhmk31.dfs.core.windows.net/product_subcategories")

##### products

In [0]:
df_pro=(df_pro
    .withColumn('ProductColor',when(trim(col('ProductColor')) =='NA', None).otherwise(col('ProductColor')))
    .withColumn('ProductSize',when(trim(col('ProductSize')) =='0', None).otherwise(col('ProductSize')))
    .withColumn('ProductStyle',when(trim(col('ProductStyle'))=='0', None).otherwise(col('ProductStyle')))
    .withColumn('ProductDescription',trim(col('ProductDescription')))
)

In [0]:
df_pro= df_pro.withColumn("ProductPrice", col("ProductPrice").cast("double"))

In [0]:
from pyspark.sql.functions import col

df_product_full = (
    df_pro
    .join(df_subcat, "ProductSubcategoryKey", "left")
    .join(df_procat, "ProductCategoryKey", "left")
    .select(
        "ProductKey",
        "ProductName",
        col("ProductPrice"))
    .dropDuplicates()
)

In [0]:
df_product_full.write.format('parquet').mode('overwrite').save("abfss://silver@adwhmk31.dfs.core.windows.net/products")

##### returns

In [0]:
df_ret= df_ret.withColumn("ReturnDate", to_date(col("ReturnDate"), "yyyy-MM-dd")).dropDuplicates()

In [0]:
df_ret.write.format('parquet').mode('overwrite').save("abfss://silver@adwhmk31.dfs.core.windows.net/returns")

##### territories

In [0]:
df_ter=df_ter.withColumnRenamed("SalesTerritoryKey", "TerritoryKey").dropDuplicates()

In [0]:
df_ter.write.format('parquet').mode('overwrite').save("abfss://silver@adwhmk31.dfs.core.windows.net/territories")

##### product categories

In [0]:
df_procat.write.format('parquet').mode('overwrite').save("abfss://silver@adwhmk31.dfs.core.windows.net/product_categories")

##### sales

In [0]:
df_sales = (
    df_sales
    .withColumn("OrderDate", to_date(col("OrderDate"), "M/d/yyyy"))
    .withColumn("StockDate", to_date(col("StockDate"), "M/d/yyyy"))
)

In [0]:
df_sales = (
    df_sales
    .withColumn("OrderQuantity", col("OrderQuantity").cast("int"))
)


In [0]:
df_sales = (
    df_sales
    .join(df_product_full.select("ProductKey", "ProductPrice"), "ProductKey", "left")
    .withColumn("Revenue", col("OrderQuantity") * col("ProductPrice"))
    .dropDuplicates()
)

In [0]:
df_sales.write.format('parquet').mode('overwrite').save("abfss://silver@adwhmk31.dfs.core.windows.net/sales")